# Cronbach's Alpha Reliability Analysis

This notebook computes Cronbach's alpha for the Likert-scale constructs used in the blended learning thesis project.

The analysis uses the final cleaned dataset:

```text
data/processed/cleaned_data.csv
```

The output files are saved to:

```text
data/processed/reliability/
```

The overall 33-item reliability analysis includes all ordinal Likert-scale variables. The item `tech_issues_freq` is reverse-coded before calculating the overall reliability because higher original values indicate more frequent technical issues, while most other Likert-scale items are positively oriented.


## 1. Import libraries and set paths

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# This notebook is designed to be placed inside the notebook/ folder.
DATA_PATH = Path("../data/processed/cleaned_data.csv")
OUTPUT_DIR = Path("../data/processed/reliability")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)


Dataset path: ..\data\processed\cleaned_data.csv
Output directory: ..\data\processed\reliability


## 2. Define helper functions

In [3]:
def cronbach_alpha(data: pd.DataFrame):
    """
    Compute Cronbach's alpha for Likert-scale items.

    Parameters
    ----------
    data : pd.DataFrame
        DataFrame containing only the items used in one scale or construct.

    Returns
    -------
    alpha : float
        Cronbach's alpha value.
    n_valid : int
        Number of complete valid responses used.
    k_items : int
        Number of items used.
    """

    data = data.apply(pd.to_numeric, errors="coerce")
    data = data.dropna(axis=0, how="any")

    n_valid = len(data)
    k_items = data.shape[1]

    if k_items < 2 or n_valid == 0:
        return np.nan, n_valid, k_items

    item_variances = data.var(axis=0, ddof=1)
    total_score = data.sum(axis=1)
    total_variance = total_score.var(ddof=1)

    if total_variance == 0:
        return np.nan, n_valid, k_items

    alpha = (k_items / (k_items - 1)) * (
        1 - item_variances.sum() / total_variance
    )

    return alpha, n_valid, k_items


def interpret_alpha(alpha: float) -> str:
    """
    Interpret Cronbach's alpha using common academic thresholds.
    """

    if pd.isna(alpha):
        return "Not available"
    elif alpha >= 0.90:
        return "Excellent"
    elif alpha >= 0.80:
        return "Good"
    elif alpha >= 0.70:
        return "Acceptable"
    elif alpha >= 0.60:
        return "Questionable / moderate"
    else:
        return "Low / questionable"


def alpha_if_item_deleted(data: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Cronbach's alpha after deleting each item one by one.
    This helps identify whether one item reduces the reliability of a construct.
    """

    results = []

    for item in data.columns:
        remaining_items = [col for col in data.columns if col != item]
        alpha, n_valid, k_items = cronbach_alpha(data[remaining_items])

        results.append(
            {
                "Deleted Item": item,
                "Remaining Items": k_items,
                "Valid Responses": n_valid,
                "Alpha if Item Deleted": round(alpha, 3)
                if not pd.isna(alpha)
                else np.nan,
            }
        )

    return pd.DataFrame(results)


## 3. Load the cleaned dataset

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please place this notebook in the notebook/ folder, or adjust DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (588, 87)


,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,career_preparation,ideal_balance,prefer_more_blended,open_strengths,open_challenges_suggestions,survey_start,survey_end,response_time_minutes,student_id,flag_speeder
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,2,"More Online than Face-to-Face (e.g., 40% In-pe...",No,NaN,NaN,2026-03-12 00:35:35.641,2026-03-12 00:37:43.468,2.130450,e20210528,True
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,3,"More Online than Face-to-Face (e.g., 40% In-pe...",Neutral/Unsure,Nothing,Nothing,2026-03-17 18:53:40.435,2026-03-17 18:56:16.257,2.597033,e20210686,True
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,4,"Balanced Half and Half (50% In-person, 50% Onl...",Neutral/Unsure,"Very good, excellent","No big challenge, i’m the best",2026-03-09 15:33:21.321,2026-03-09 15:36:31.387,3.167767,e20241146,False
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,3,"Mostly Face-to-Face (e.g., 80% In-person, 20% ...",Neutral/Unsure,Will try hard,Lack of self-discipline,2026-03-09 15:32:15.225,2026-03-09 15:35:36.121,3.348267,e20240609,False
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,3,"Balanced Half and Half (50% In-person, 50% Onl...",Neutral/Unsure,getting more experience,Discipline on daily studying,2026-03-09 15:31:35.190,2026-03-09 15:35:12.175,3.616417,e20240542,False


## 4. Define Likert-scale constructs

The constructs below match the major dimensions used in the thesis analysis: lecturer support, perceived benefits, learning material use, self-regulation, and engagement/interaction.


In [5]:
constructs = {
    "Lecturer Support": [
        "lect_clear_instructions",
        "lect_responsive",
        "lect_diverse_tools",
        "lect_timely_feedback",
        "lect_foster_interaction",
    ],
    "Perceived Benefits": [
        "benefit_flexibility",
        "benefit_variety",
        "benefit_recorded_access",
        "benefit_self_study_time",
        "benefit_life_balance",
        "benefit_self_directed",
    ],
    "Learning Material Use": [
        "use_lecture_slides",
        "use_video_lectures",
        "use_quizzes",
        "use_articles",
        "use_forums",
        "use_simulations",
    ],
    "Self-Regulation": [
        "self_prioritize_deadlines",
        "self_study_schedule",
        "self_prepare_class",
        "self_responsibility",
    ],
    "Engagement and Interaction": [
        "online_discussion_participation",
        "peer_collaboration",
        "comfort_asking_questions",
        "sense_of_community",
    ],
}

all_33_likert_items = [
    "video_helpfulness",
    "digital_literacy_improvement",
    "use_lecture_slides",
    "use_video_lectures",
    "use_quizzes",
    "use_articles",
    "use_forums",
    "use_simulations",
    "online_discussion_participation",
    "peer_collaboration",
    "comfort_asking_questions",
    "sense_of_community",
    "integration_quality",
    "benefit_flexibility",
    "benefit_variety",
    "benefit_recorded_access",
    "benefit_self_study_time",
    "benefit_life_balance",
    "benefit_self_directed",
    "overall_understanding",
    "lect_clear_instructions",
    "lect_responsive",
    "lect_diverse_tools",
    "lect_timely_feedback",
    "lect_foster_interaction",
    "self_prioritize_deadlines",
    "self_study_schedule",
    "self_prepare_class",
    "self_responsibility",
    "overall_satisfaction",
    "career_preparation",
    "tech_issues_freq",
    "lms_usability",
]

print("Number of Likert items:", len(all_33_likert_items))


Number of Likert items: 33


## 5. Check that all required columns exist

In [6]:
required_columns = set(all_33_likert_items)

for items in constructs.values():
    required_columns.update(items)

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(
        "The following required columns are missing from the cleaned dataset:\n"
        + "\n".join(missing_columns)
    )

print("All required Likert-scale columns are available.")


All required Likert-scale columns are available.


## 6. Reverse-code `tech_issues_freq`

The original `tech_issues_freq` item is negatively oriented:

- Higher original value = more frequent technical issues
- Most other Likert items: higher value = more positive experience

Therefore, it is reverse-coded before being included in the overall 33-item Cronbach's alpha calculation.


In [7]:
df_alpha = df.copy()

df_alpha["tech_issues_freq"] = pd.to_numeric(
    df_alpha["tech_issues_freq"], errors="coerce"
)

df_alpha["tech_issues_freq_reversed"] = 6 - df_alpha["tech_issues_freq"]

all_33_likert_items_for_alpha = [
    "tech_issues_freq_reversed" if col == "tech_issues_freq" else col
    for col in all_33_likert_items
]

df_alpha[["tech_issues_freq", "tech_issues_freq_reversed"]].head()


,tech_issues_freq,tech_issues_freq_reversed
0,2,4
1,3,3
2,3,3
3,4,2
4,3,3


## 7. Compute Cronbach's alpha

In [8]:
results = []

# Overall 33-item reliability
overall_alpha, overall_n, overall_k = cronbach_alpha(
    df_alpha[all_33_likert_items_for_alpha]
)

results.append(
    {
        "Construct": "Overall Likert-Scale Item Pool",
        "Number of Items": overall_k,
        "Valid Responses": overall_n,
        "Cronbach Alpha": round(overall_alpha, 3)
        if not pd.isna(overall_alpha)
        else np.nan,
        "Interpretation": interpret_alpha(overall_alpha),
    }
)

# Construct-level reliability
for construct_name, items in constructs.items():
    construct_data = df[items]

    alpha, n_valid, k_items = cronbach_alpha(construct_data)

    results.append(
        {
            "Construct": construct_name,
            "Number of Items": k_items,
            "Valid Responses": n_valid,
            "Cronbach Alpha": round(alpha, 3)
            if not pd.isna(alpha)
            else np.nan,
            "Interpretation": interpret_alpha(alpha),
        }
    )

results_df = pd.DataFrame(results)
results_df


,Construct,Number of Items,Valid Responses,Cronbach Alpha,Interpretation
0,Overall Likert-Scale Item Pool,33,588,0.881,Good
1,Lecturer Support,5,588,0.814,Good
2,Perceived Benefits,6,588,0.773,Acceptable
3,Learning Material Use,6,588,0.624,Questionable / moderate
4,Self-Regulation,4,588,0.606,Questionable / moderate
5,Engagement and Interaction,4,588,0.570,Low / questionable


## 8. Compute alpha-if-item-deleted diagnostics

In [9]:
# Overall 33-item alpha-if-item-deleted
overall_deleted = alpha_if_item_deleted(
    df_alpha[all_33_likert_items_for_alpha]
)

overall_deleted.to_csv(
    OUTPUT_DIR / "alpha_if_item_deleted_overall_33_items.csv",
    index=False,
    encoding="utf-8-sig",
)

# Construct-level alpha-if-item-deleted outputs
for construct_name, items in constructs.items():
    deleted_result = alpha_if_item_deleted(df[items])

    safe_name = (
        construct_name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
    )

    deleted_result.to_csv(
        OUTPUT_DIR / f"alpha_if_item_deleted_{safe_name}.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Alpha-if-item-deleted diagnostics saved.")
overall_deleted.head()


Alpha-if-item-deleted diagnostics saved.


,Deleted Item,Remaining Items,Valid Responses,Alpha if Item Deleted
0,video_helpfulness,32,588,0.879
1,digital_literacy_improvement,32,588,0.876
2,use_lecture_slides,32,588,0.878
3,use_video_lectures,32,588,0.878
4,use_quizzes,32,588,0.879


## 9. Save reliability results for thesis appendix

In [10]:
results_csv_path = OUTPUT_DIR / "cronbach_alpha_results.csv"
results_tex_path = OUTPUT_DIR / "cronbach_alpha_table.tex"

results_df.to_csv(
    results_csv_path,
    index=False,
    encoding="utf-8-sig",
)

latex_table = results_df.to_latex(
    index=False,
    escape=False,
    caption="Cronbach's Alpha Reliability Results for Likert-Scale Constructs",
    label="tab:cronbach-alpha",
    column_format="lcccl",
)

with open(results_tex_path, "w", encoding="utf-8") as f:
    f.write(latex_table)

print("Saved output files:")
print(f"- {results_csv_path}")
print(f"- {results_tex_path}")
print(f"- {OUTPUT_DIR / 'alpha_if_item_deleted_overall_33_items.csv'}")


Saved output files:
- ..\data\processed\reliability\cronbach_alpha_results.csv
- ..\data\processed\reliability\cronbach_alpha_table.tex
- ..\data\processed\reliability\alpha_if_item_deleted_overall_33_items.csv


## 10. Thesis interpretation

Use this wording in your methodology or appendix after confirming the generated table values:

```latex
To assess the reliability of the Likert-scale items used in the survey, Cronbach's alpha was computed using the final cleaned dataset. The reliability analysis was conducted after preprocessing, using the 33 ordinal Likert-scale variables that were later used for multivariate analysis and clustering. Since the cleaned dataset contained no missing values in these 33 variables, all 588 valid responses were included in the reliability calculation.

For the overall 33-item reliability analysis, the item \texttt{tech\_issues\_freq} was reverse-coded before computing Cronbach's alpha because higher original values indicated more frequent technical problems, whereas most other Likert-scale items were positively oriented.

The overall Cronbach's alpha for the 33-item Likert-scale pool was \(\alpha = 0.881\), indicating good internal consistency. Construct-level reliability was also examined because the questionnaire covered multiple dimensions of blended learning experience. The lecturer support construct showed good reliability, while the perceived benefits construct showed acceptable reliability. Other constructs, including learning material use, self-regulation, and engagement and interaction, showed lower reliability values and were therefore interpreted cautiously. These results indicate that the Likert-scale items were suitable for exploratory analysis, clustering, and student profile interpretation, but the questionnaire should not be treated as a fully validated psychometric instrument.
```
